In [ ]:
# Training config — injected at submit time by KaggleTrainingAdapter
CONFIG = {{config}}
print('Experiment:', CONFIG['experiment_name'])
print('Model     :', CONFIG['model'])
print('Epochs    :', CONFIG['epochs'])

In [ ]:
import subprocess, sys, re
from pathlib import Path

dataset_slug = CONFIG['experiment_name'] + '-data'
input_base = Path('/kaggle/input') / dataset_slug

# Test whether the pre-installed torch can actually run a CUDA kernel.
# If not (cudaErrorNoKernelImageForDevice), reinstall matching the driver's CUDA version.
cuda_test = subprocess.run(
    [sys.executable, '-c',
     'import torch; t=torch.zeros(1).cuda(); print(f"CUDA OK gpu={torch.cuda.get_device_name(0)}")'],
    capture_output=True, text=True
)
if 'CUDA OK' in cuda_test.stdout:
    print(cuda_test.stdout.strip())
else:
    print("CUDA test failed:", (cuda_test.stderr or cuda_test.stdout).strip()[:300])
    smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    m = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', smi.stdout)
    if m:
        ver = int(m.group(1)) * 100 + int(m.group(2))
        cu_tag = 'cu128' if ver >= 1208 else 'cu124' if ver >= 1204 else 'cu121'
    else:
        cu_tag = 'cu124'
    print(f"Reinstalling torch from {cu_tag} ...")
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall',
        'torch', '--index-url', f'https://download.pytorch.org/whl/{cu_tag}'
    ], check=True)
    verify = subprocess.run(
        [sys.executable, '-c',
         'import torch; t=torch.zeros(1).cuda(); print(f"After reinstall: {torch.cuda.get_device_name(0)} torch={torch.__version__}")'],
        capture_output=True, text=True
    )
    print(verify.stdout.strip() or verify.stderr.strip())

# Install project wheel (no deps — training deps are pre-installed on Kaggle)
wheels = list(input_base.glob('*.whl'))
if not wheels:
    raise FileNotFoundError(f'No .whl found in {input_base} — re-run the training trigger to rebuild the dataset.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-q', str(wheels[0])], check=True)
print(f'Installed {wheels[0].name}.')

In [ ]:
import shutil
from pathlib import Path

# Kaggle mounts attached datasets at /kaggle/input/<dataset-slug>/
dataset_slug = CONFIG['experiment_name'] + '-data'
input_base = Path('/kaggle/input') / dataset_slug

# Copy flat .jsonl files from the dataset into data/
data_dst = Path('data')
data_dst.mkdir(exist_ok=True)
for jsonl in input_base.glob('*.jsonl'):
    shutil.copy(jsonl, data_dst / jsonl.name)
    print(f'Copied {jsonl.name} ({jsonl.stat().st_size // 1000} KB)')

In [ ]:
import subprocess, sys

# Package is installed, so cli.train is importable as a module directly
cmd = [
    sys.executable, '-m', 'cli.train',
    '--model', CONFIG['model'],
    '--epochs', str(CONFIG['epochs']),
    '--patience', str(CONFIG['patience']),
    '--warmup-ratio', str(CONFIG['warmup_ratio']),
    '--train-data', 'data/train.jsonl',
    '--eval-data', 'data/eval.jsonl',
    '--output-dir', 'models/checkpoints',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Training complete.')

In [ ]:
import tarfile
from pathlib import Path

checkpoint_dir = Path('models/checkpoints')
output_archive = Path('/kaggle/working/checkpoint.tar.gz')

if not checkpoint_dir.exists():
    raise FileNotFoundError(f'Checkpoint not found at {checkpoint_dir}')

with tarfile.open(output_archive, 'w:gz') as tf:
    tf.add(checkpoint_dir, arcname='checkpoints')

print(f'Packaged -> {output_archive} ({output_archive.stat().st_size / 1e6:.1f} MB)')